In [1]:
# lectura de los datos

import pandas as pd
import read_data

# Suponiendo que el activo tiene ese nombre
# solo es remplazar por el nombre que se tenga
# solo se acepta archivos .parquet y .csv
nombre_activo: str = "EURUSD.parquet"
data: pd.DataFrame = read_data.read_asset(nombre_activo)

primer_lunes = data[data.index.dayofweek == 0].index[0]

data = data[primer_lunes:]

In [2]:
import find_best
import keys
import pandas as pd
import read_data
from use_tecnics import simple_methods, main

keys.candles = 20
keys.methods = simple_methods

ventana_entrenamiento = pd.Timedelta(weeks=3) + pd.Timedelta(days=4)
inicio_testeo = pd.Timedelta(days=3)
fin_testeo = pd.Timedelta(days=4)

inicio_train = data[data.index.dayofweek == 0].index[0].normalize()
fin_datos = data.index[-1]

signals_and_prices = None

while True:
    train_time = inicio_train + ventana_entrenamiento
    inicio_test = train_time + inicio_testeo
    fin_test = inicio_test + fin_testeo

    if fin_test >= fin_datos:
        break

    print(f"Entrenamiento: {inicio_train.strftime('%Y-%m-%d')} a {train_time.strftime('%Y-%m-%d')}")

    sub_data = data[inicio_train: train_time]
    best_ma = find_best.opti_main(sub_data)

    minutos_de_colchon = best_ma[1] * best_ma[2] * (1 if best_ma[0] in simple_methods else 8)
    tiempo_colchon = train_time - pd.Timedelta(minutes=minutos_de_colchon)
    
    bloque_completo = pd.concat([data[tiempo_colchon: train_time - pd.Timedelta(minutes=1)],
                                 data[inicio_test: fin_test]])

    velas_cerradas = read_data.ohlc_form(bloque_completo, f"{best_ma[1]}min")["close"]

    señales_generadas = main(best_ma[0], velas_cerradas, best_ma[2:])

    test_signals_only = señales_generadas.loc[inicio_test : fin_test]
    
    if signals_and_prices is None:
        signals_and_prices = test_signals_only
    else:
        signals_and_prices = pd.concat([signals_and_prices, test_signals_only])

    inicio_train = inicio_train + pd.Timedelta(days=7)

Entrenamiento: 2025-02-24 a 2025-03-21
Resultado obtenido entrenando desde 2025-02-24 hasta 2025-03-20
Método: TEMA, Datos optimizados [np.int64(16), np.int64(66)]

hit ratio: 0.47398843930635837
risk reward: 0.8067011486900163
profit factor: 0.7269175185997949
trades: 173
Resultado de sobre ajuste -1.3546642711379933
Entrenamiento: 2025-03-03 a 2025-03-28
Resultado obtenido entrenando desde 2025-03-03 hasta 2025-03-27
Método: MIDPOINT, Datos optimizados [np.int64(16), np.int64(66)]

hit ratio: 0.43902439024390244
risk reward: 0.7910825957452918
profit factor: 0.6191081184093588
trades: 123
Resultado de sobre ajuste -1.51220615955591
Entrenamiento: 2025-03-10 a 2025-04-04
Resultado obtenido entrenando desde 2025-03-10 hasta 2025-04-03
Método: MIDPOINT, Datos optimizados [np.int64(16), np.int64(66)]

hit ratio: 0.5
risk reward: 1.080506895266493
profit factor: 1.080506895266493
trades: 138
Resultado de sobre ajuste -1.6027963861484276
Entrenamiento: 2025-03-17 a 2025-04-11
Resultado obt

KeyboardInterrupt: 

In [15]:
signals_and_prices = pd.read_parquet("Data/historical_signals_and_prices.parquet")
signals_and_prices

,Signals,Prices
time,,
2025-03-24 02:56:00,-1.0,1.08367
2025-03-24 04:32:00,1.0,1.08235
2025-03-24 05:04:00,-1.0,1.08269
2025-03-24 05:20:00,1.0,1.08309
2025-03-24 05:36:00,-1.0,1.08264
...,...,...
2026-01-08 10:08:00,1.0,1.16743
2026-01-08 12:16:00,-1.0,1.16788
2026-01-08 12:32:00,1.0,1.16776


In [20]:
t = signals_and_prices.index[0]
print(signals_and_prices["Prices"][t])

1.08367


In [21]:
cantidad_comprada = 0
initial_mon = 1000
for index in signals_and_prices:
    precio_actual = signals_and_prices["Prices"][index]
    signal = signals_and_prices["Signals"][index]

    if signal == 1 and cantidad_comprada == 0:
        cantidad_comprada = int(initial_mon / precio_actual)

        if cantidad_comprada == 0:
            continue

        precio_compra = cantidad_comprada * precio_actual
        initial_mon -= precio_compra 

    elif signal == -1 and cantidad_comprada != 0:
            cantidad_ganada = precio_actual * cantidad_comprada
            initial_mon += cantidad_ganada
            cantidad_comprada = 0

print(initial_mon)

KeyError: 'Signals'